In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
!pip install -q datasets pandas

In [6]:
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

CHUNKS_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/chunks/chunks'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Take only first 10 chunk files
chunk_files = sorted([f for f in os.listdir(CHUNKS_DIR) if f.endswith('.jsonl')])[:10]
print(f"Using {len(chunk_files)} chunk files")

# Load
print("Loading chunks...")
all_chunks = []
for f in tqdm(chunk_files):
    with open(f'{CHUNKS_DIR}/{f}') as fh:
        for line in fh:
            all_chunks.append(json.loads(line))

df = pd.DataFrame(all_chunks)
print(f"Loaded {len(df):,} total chunks")

# Keep audio aspects
AUDIO_ASPECTS = ['bass', 'treble', 'mids', 'soundstage', 'comfort', 'comparison']
audio_df = df[df['aspect'].isin(AUDIO_ASPECTS)].copy()

print(f"Audio chunks: {len(audio_df):,}")
for aspect, count in audio_df['aspect'].value_counts().items():
    print(f"  {aspect}: {count:,}")

audio_df.to_json(f'{OUTPUT_DIR}/audio_chunks.json', orient='records')
print(f"\n✓ Saved")

Using 10 chunk files
Loading chunks...


100%|██████████| 10/10 [00:03<00:00,  3.10it/s]


Loaded 654,269 total chunks
Audio chunks: 111,742
  comfort: 53,033
  bass: 28,595
  comparison: 13,766
  treble: 8,483
  mids: 5,523
  soundstage: 2,342

✓ Saved


In [7]:
!pip install -q sentence-transformers faiss-gpu

from sentence_transformers import SentenceTransformer
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = '/kaggle/working/embeddings'
audio_df = pd.read_json(f'{OUTPUT_DIR}/audio_chunks.json')
texts = audio_df['chunk_text'].tolist()
print(f"Embedding {len(texts):,} chunks...\n")

model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

BATCH_SIZE = 512
embeddings_list = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list).astype('float32')
np.save(f'{OUTPUT_DIR}/audio_embeddings.npy', embeddings)
print(f"\n✓ Shape: {embeddings.shape}")

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


AssertionError: Torch not compiled with CUDA enabled

In [8]:
!pip install -q sentence-transformers faiss-cpu

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print("Using CPU - embedding will take 10-15 min for 10 files")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 18.8 MB/s eta 0:00:0000:0100:01
CUDA available: False
Using CPU - embedding will take 10-15 min for 10 files


In [9]:
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

CHUNKS_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/chunks/chunks'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

chunk_files = sorted([f for f in os.listdir(CHUNKS_DIR) if f.endswith('.jsonl')])[:10]
print(f"Using {len(chunk_files)} files")

all_chunks = []
for f in tqdm(chunk_files):
    with open(f'{CHUNKS_DIR}/{f}') as fh:
        for line in fh:
            all_chunks.append(json.loads(line))

df = pd.DataFrame(all_chunks)
AUDIO_ASPECTS = ['bass', 'treble', 'mids', 'soundstage', 'comfort', 'comparison']
audio_df = df[df['aspect'].isin(AUDIO_ASPECTS)].copy()
print(f"Audio chunks: {len(audio_df):,}")

audio_df.to_json(f'{OUTPUT_DIR}/audio_chunks.json', orient='records')
print("✓ Saved")

Using 10 files


100%|██████████| 10/10 [00:03<00:00,  2.88it/s]


Audio chunks: 111,742
✓ Saved


In [10]:
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = '/kaggle/working/embeddings'
audio_df = pd.read_json(f'{OUTPUT_DIR}/audio_chunks.json')
texts = audio_df['chunk_text'].tolist()
print(f"Embedding {len(texts):,} chunks on CPU...\n")

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

BATCH_SIZE = 128
embeddings_list = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list).astype('float32')
np.save(f'{OUTPUT_DIR}/audio_embeddings.npy', embeddings)
print(f"\n✓ Shape: {embeddings.shape}")

Embedding 111,742 chunks on CPU...



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100%|██████████| 873/873 [42:53<00:00,  2.95s/it]



✓ Shape: (111742, 384)


In [11]:
import faiss

OUTPUT_DIR = '/kaggle/working/embeddings'

embeddings = np.load(f'{OUTPUT_DIR}/audio_embeddings.npy')
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"✓ Index: {index.ntotal:,} vectors")

audio_df.to_csv(f'{OUTPUT_DIR}/chunk_metadata.csv', index_label='chunk_id')

# Test
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
query = "warm IEMs with good bass for jazz"
q_vec = model.encode([query])
faiss.normalize_L2(q_vec)
scores, indices = index.search(q_vec, 5)

print(f"\nQuery: '{query}'")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    row = audio_df.iloc[idx]
    print(f"\n  #{i+1} [{score:.3f}] [{row['aspect']}]")
    print(f"  {row['chunk_text'][:150]}...")

print("\n✓ Done!")

✓ Index: 111,742 vectors


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: 'warm IEMs with good bass for jazz'

  #1 [0.680] [bass]
  Fantastic IEMs! I am a bass player and singer and these are exactly what I was looking for.<br />I previously used a set of Audio Technica ATH-IM50 an...

  #2 [0.664] [bass]
  I play a 5 string bass for church. These IEM's are great. At least for bass, you need to use the ear pieces that are the expand foam like ear plugs, s...

  #3 [0.654] [bass]
  No, you're not buying these for bass. But there is enough bass for these to sound warm alongside its sparkly highs. Every instrument feels well repres...

  #4 [0.644] [bass]
  They have a very mellow bass which is great for listening to jazz....

  #5 [0.640] [bass]
  I looked at a lot of iems before I decided on these and I am very happy with my choice.<br />OK, it takes a little bit to get used to putting them in,...

✓ Done!


In [12]:
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import re

OUTPUT_DIR = '/kaggle/working/embeddings'

# Load
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
index = faiss.read_index(f'{OUTPUT_DIR}/faiss_audio_index.bin')
audio_df = pd.read_csv(f'{OUTPUT_DIR}/chunk_metadata.csv')

def clean_text(text):
    """Remove HTML tags and clean up text"""
    text = re.sub(r'<br\s*/?>', ' ', text)  # Remove <br>
    text = re.sub(r'<[^>]+>', '', text)      # Remove other HTML
    text = ' '.join(text.split())             # Clean whitespace
    return text

def search(query, k=5):
    """Search and return formatted results"""
    q_vec = model.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = audio_df.iloc[idx]
        results.append({
            'score': round(float(score), 3),
            'aspect': row['aspect'],
            'sentiment': row['sentiment'],
            'rating': row['rating'],
            'product': str(row['title'])[:80],
            'text': clean_text(str(row['chunk_text']))
        })
    return results

# Test multiple queries
queries = [
    "warm IEMs with good bass for jazz music",
    "comfortable headphones for long sessions",
    "best budget earphones under $50",
    "IEMs with wide soundstage and detailed treble",
]

for query in queries:
    print(f"\n{'='*70}")
    print(f"🔍 Query: {query}")
    print(f"{'='*70}")
    
    results = search(query, k=5)
    
    for i, r in enumerate(results):
        print(f"\n  #{i+1} │ Score: {r['score']} │ {r['aspect']} │ {r['sentiment']} │ ⭐{r['rating']}")
        print(f"  Product: {r['product']}")
        print(f"  {r['text'][:200]}")
        if len(r['text']) > 200:
            print(f"  ...")
    
    print()

print("="*70)
print("✅ Search working!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /kaggle/working/embeddings/faiss_audio_index.bin for reading: No such file or directory

In [13]:
import pandas as pd
import numpy as np
import json
import os
import re
from tqdm import tqdm
import faiss
from sentence_transformers import SentenceTransformer

# ============================================
# SETUP
# ============================================
CHUNKS_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/chunks/chunks'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# LOAD CHUNKS
# ============================================
print("Loading chunks...")
chunk_files = sorted([f for f in os.listdir(CHUNKS_DIR) if f.endswith('.jsonl')])[:10]
all_chunks = []
for f in tqdm(chunk_files):
    with open(f'{CHUNKS_DIR}/{f}') as fh:
        for line in fh:
            all_chunks.append(json.loads(line))

df = pd.DataFrame(all_chunks)
AUDIO_ASPECTS = ['bass', 'treble', 'mids', 'soundstage', 'comfort', 'comparison']
audio_df = df[df['aspect'].isin(AUDIO_ASPECTS)].copy()
print(f"Audio chunks: {len(audio_df):,}")

# ============================================
# EMBED
# ============================================
print("\nLoading model...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
texts = audio_df['chunk_text'].tolist()

print(f"Embedding {len(texts):,} chunks...")
BATCH_SIZE = 128
embeddings_list = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list).astype('float32')
print(f"✓ Embeddings: {embeddings.shape}")

# ============================================
# BUILD INDEX
# ============================================
print("\nBuilding FAISS index...")
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Save
faiss.write_index(index, f'{OUTPUT_DIR}/faiss_audio_index.bin')
audio_df.to_csv(f'{OUTPUT_DIR}/chunk_metadata.csv', index_label='chunk_id')
print(f"✓ Index saved: {index.ntotal:,} vectors")

# ============================================
# TEST
# ============================================
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    return ' '.join(text.split())

print("\n" + "="*60)
print("TESTING")
print("="*60)

queries = [
    "warm IEMs with good bass for jazz",
    "comfortable headphones for long sessions",
    "best budget earphones under $50",
]

for query in queries:
    print(f"\n🔍 {query}")
    
    q_vec = model.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, 5)
    
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        row = audio_df.iloc[idx]
        text = clean_text(row['chunk_text'])[:150]
        print(f"  #{i+1} [{score:.3f}] {row['aspect']} | ⭐{row['rating']} | {text}...")

print("\n✅ Done!")

Loading chunks...


100%|██████████| 10/10 [00:03<00:00,  2.89it/s]


Audio chunks: 111,742

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 111,742 chunks...


  1%|          | 8/873 [00:24<44:36,  3.09s/it]


KeyboardInterrupt: 

In [14]:
import os

OUTPUT_DIR = '/kaggle/working/embeddings'

print("Files in embeddings folder:")
if os.path.exists(OUTPUT_DIR):
    for f in os.listdir(OUTPUT_DIR):
        size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024 / 1024
        print(f"  {size:.1f} MB - {f}")
else:
    print("  Folder doesn't exist!")

Files in embeddings folder:
  163.7 MB - audio_embeddings.npy
  41.3 MB - audio_chunks.json
  32.0 MB - chunk_metadata.csv


In [15]:
import numpy as np
import faiss
import os

OUTPUT_DIR = '/kaggle/working/embeddings'

# Load existing embeddings
print("Loading embeddings...")
embeddings = np.load(f'{OUTPUT_DIR}/audio_embeddings.npy')
print(f"Shape: {embeddings.shape}")

# Normalize and build index
print("Building index...")
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Save index
faiss.write_index(index, f'{OUTPUT_DIR}/faiss_audio_index.bin')
size_mb = os.path.getsize(f'{OUTPUT_DIR}/faiss_audio_index.bin') / 1024 / 1024
print(f"✓ Index saved: {index.ntotal:,} vectors ({size_mb:.0f} MB)")

Loading embeddings...
Shape: (111742, 384)
Building index...
✓ Index saved: 111,742 vectors (164 MB)


In [16]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = '/kaggle/working/embeddings'

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
index = faiss.read_index(f'{OUTPUT_DIR}/faiss_audio_index.bin')
audio_df = pd.read_csv(f'{OUTPUT_DIR}/chunk_metadata.csv')

def clean(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    return ' '.join(text.split())

query = "warm IEMs with good bass for jazz"
q_vec = model.encode([query])
faiss.normalize_L2(q_vec)
scores, indices = index.search(q_vec, 5)

print(f"🔍 {query}\n")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    row = audio_df.iloc[idx]
    print(f"#{i+1} [{score:.3f}] {row['aspect']} | ⭐{row['rating']}")
    print(f"  {clean(row['chunk_text'])[:200]}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 warm IEMs with good bass for jazz

#1 [0.680] bass | ⭐5
  Fantastic IEMs! I am a bass player and singer and these are exactly what I was looking for. I previously used a set of Audio Technica ATH-IM50 and while they are great, these were a big step up in all

#2 [0.664] bass | ⭐5
  I play a 5 string bass for church. These IEM's are great. At least for bass, you need to use the ear pieces that are the expand foam like ear plugs, so you get a good seal.

#3 [0.654] bass | ⭐5
  No, you're not buying these for bass. But there is enough bass for these to sound warm alongside its sparkly highs. Every instrument feels well represented with focus on the high frequencies.

#4 [0.644] bass | ⭐4
  They have a very mellow bass which is great for listening to jazz.

#5 [0.640] bass | ⭐5
  I looked at a lot of iems before I decided on these and I am very happy with my choice. OK, it takes a little bit to get used to putting them in, but it is well worth it. The sound is fantastic. The b



In [17]:
import pandas as pd

OUTPUT_DIR = '/kaggle/working/embeddings'
audio_df = pd.read_csv(f'{OUTPUT_DIR}/chunk_metadata.csv')

# Show unique products
print(f"Total unique products: {audio_df['parent_asin'].nunique():,}")
print(f"Total chunks: {len(audio_df):,}")

# Show product titles
print(f"\nSample product titles:")
titles = audio_df['title'].value_counts().head(20)
for title, count in titles.items():
    print(f"  [{count} chunks] {str(title)[:80]}")

# Show aspect distribution
print(f"\nAspect distribution:")
for aspect, count in audio_df['aspect'].value_counts().items():
    print(f"  {aspect}: {count:,}")

Total unique products: 8,929
Total chunks: 111,742

Sample product titles:
  [365 chunks] Great product
  [283 chunks] Great sound
  [269 chunks] Perfect
  [260 chunks] Perfect fit
  [207 chunks] Great
  [205 chunks] Works great
  [202 chunks] Comfortable
  [179 chunks] Awesome
  [179 chunks] Great value
  [175 chunks] Good product
  [151 chunks] Good
  [132 chunks] Great headphones
  [132 chunks] Disappointed
  [123 chunks] Great!
  [118 chunks] Excellent
  [116 chunks] Good quality
  [113 chunks] Great Product
  [104 chunks] Great for the price
  [102 chunks] Good value
  [99 chunks] Nice

Aspect distribution:
  comfort: 53,033
  bass: 28,595
  comparison: 13,766
  treble: 8,483
  mids: 5,523
  soundstage: 2,342


In [18]:
import pandas as pd
import re
import faiss
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = '/kaggle/working/embeddings'

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
index = faiss.read_index(f'{OUTPUT_DIR}/faiss_audio_index.bin')
audio_df = pd.read_csv(f'{OUTPUT_DIR}/chunk_metadata.csv')

def clean(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    return ' '.join(text.split())

query = "warm IEMs with good bass for jazz"
q_vec = model.encode([query])
faiss.normalize_L2(q_vec)
scores, indices = index.search(q_vec, 5)

print(f"🔍 {query}\n")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    row = audio_df.iloc[idx]
    print(f"#{i+1} [{score:.3f}] {row['aspect']} | ⭐{row['rating']}")
    print(f"  Product: {row['title'][:100]}")
    print(f"  Text: {clean(row['chunk_text'])[:200]}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 warm IEMs with good bass for jazz

#1 [0.680] bass | ⭐5
  Product: Big upgrade for live playing - bass/vocals
  Text: Fantastic IEMs! I am a bass player and singer and these are exactly what I was looking for. I previously used a set of Audio Technica ATH-IM50 and while they are great, these were a big step up in all

#2 [0.664] bass | ⭐5
  Product: Used for 2 years now and love them
  Text: I play a 5 string bass for church. These IEM's are great. At least for bass, you need to use the ear pieces that are the expand foam like ear plugs, so you get a good seal.

#3 [0.654] bass | ⭐5
  Product: There is bass, and it's well represented.
  Text: No, you're not buying these for bass. But there is enough bass for these to sound warm alongside its sparkly highs. Every instrument feels well represented with focus on the high frequencies.

#4 [0.644] bass | ⭐4
  Product: VERY GOOD HEADPHONES
  Text: They have a very mellow bass which is great for listening to jazz.

#5 [0.640] bass | ⭐5
  Pr

In [2]:
!pip install -q sentence-transformers faiss-cpu

import pandas as pd
import numpy as np
import json
import os
import re
import faiss
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# Paths
CHUNKS_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/chunks/chunks'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# Load index + metadata
index = faiss.read_index(f'{OUTPUT_DIR}/faiss_audio_index.bin')
audio_df = pd.read_csv(f'{OUTPUT_DIR}/chunk_metadata.csv')

print(f"✓ Loaded: {index.ntotal:,} vectors, {len(audio_df):,} chunks")
print(f"✓ Products: {audio_df['parent_asin'].nunique():,}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.1 MB/s eta 0:00:00:00:0100:01


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /kaggle/working/embeddings/faiss_audio_index.bin for reading: No such file or directory

In [ ]:
def parse_query(query):
    """Extract structured constraints from natural language query"""
    constraints = {}
    query_lower = query.lower()
    
    # Price: "under $100", "under 100 dollars", "below $50", "budget 200"
    price_patterns = [
        r'(?:under|below|less than|budget|max|upto|up to)\s*\$?(\d+)',
        r'(?:under|below|less than|budget|max|upto|up to)\s*(\d+)\s*(?:dollars|bucks|usd)',
        r'\$(\d+)\s*(?:budget|max|limit)',
    ]
    for pattern in price_patterns:
        match = re.search(pattern, query_lower)
        if match:
            constraints['max_price'] = int(match.group(1))
            break
    
    # Product type
    if any(w in query_lower for w in ['iem', 'in-ear', 'in ear', 'iems']):
        constraints['type'] = 'IEM'
    elif any(w in query_lower for w in ['headphone', 'headset', 'over-ear', 'over ear', 'on-ear']):
        constraints['type'] = 'headphone'
    elif any(w in query_lower for w in ['earbud', 'earbuds', 'tws', 'true wireless', 'bluetooth ear']):
        constraints['type'] = 'earbud'
    
    # Sound signature
    for sig in ['warm', 'neutral', 'bright', 'dark', 'bassy', 'v-shaped', 'analytical', 'fun']:
        if sig in query_lower:
            constraints['sound_signature'] = sig
    
    # Use case
    for use in ['gaming', 'studio', 'commuting', 'gym', 'workout', 'office', 'travel', 'jazz', 'rock', 'classical']:
        if use in query_lower:
            constraints['use_case'] = use
    
    # Special: noise canceling / wireless
    if any(w in query_lower for w in ['noise cancel', 'anc', 'noise isolating']):
        constraints['noise_cancel'] = True
    if any(w in query_lower for w in ['wireless', 'bluetooth', 'bt']):
        constraints['wireless'] = True
    
    return constraints

# Test it
test_queries = [
    "warm IEMs under 100 dollars for jazz",
    "noise canceling headphones under 200 for office",
    "best budget earbuds under 50",
    "bassy IEMs for gym",
]

print("Query Parser Test:")
for q in test_queries:
    print(f"\n  Query: {q}")
    print(f"  Constraints: {parse_query(q)}")

In [3]:
!pip install -q sentence-transformers faiss-cpu

import pandas as pd
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# Your file paths
INPUT_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/embeddings'

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# Load index + metadata (instant - no rebuilding)
index = faiss.read_index(f'{INPUT_DIR}/faiss_audio_index.bin')
audio_df = pd.read_csv(f'{INPUT_DIR}/chunk_metadata.csv')

print(f"✓ Loaded: {index.ntotal:,} vectors")
print(f"✓ Products: {audio_df['parent_asin'].nunique():,}")
print(f"✓ Ready for queries!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded: 111,742 vectors
✓ Products: 8,929
✓ Ready for queries!


In [4]:
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    return ' '.join(text.split())

def parse_query(query):
    """Extract price, type, features from natural language"""
    constraints = {}
    q = query.lower()
    
    # Price: "under $100", "under 100 dollars", "below 50"
    m = re.search(r'(?:under|below|less than|budget|max|upto)\s*\$?(\d+)', q)
    if m: constraints['max_price'] = int(m.group(1))
    
    # Type
    if any(w in q for w in ['iem', 'in-ear', 'in ear']):
        constraints['type'] = 'IEM'
    elif any(w in q for w in ['headphone', 'headset', 'over-ear']):
        constraints['type'] = 'headphone'
    elif any(w in q for w in ['earbud', 'tws', 'true wireless']):
        constraints['type'] = 'earbud'
    
    # Sound signature
    for sig in ['warm', 'neutral', 'bright', 'bassy', 'v-shaped', 'dark', 'analytical']:
        if sig in q: constraints['sound_signature'] = sig
    
    # Use case
    for use in ['gaming', 'studio', 'commuting', 'gym', 'workout', 'office', 'travel', 'jazz']:
        if use in q: constraints['use_case'] = use
    
    # Features
    if any(w in q for w in ['noise cancel', 'anc', 'isolating']): constraints['noise_cancel'] = True
    if any(w in q for w in ['wireless', 'bluetooth']): constraints['wireless'] = True
    
    return constraints

def search(query, k=5):
    """Full pipeline: Parse → Filter → RAG Search → Rank"""
    constraints = parse_query(query)
    
    # Get valid products
    valid = set(audio_df['parent_asin'].unique())
    
    # Type filter
    if 'type' in constraints:
        kw = {'IEM': 'iem|in-ear', 'headphone': 'headphone|headset|over-ear', 'earbud': 'earbud|tws|true wireless'}
        m = audio_df[audio_df['chunk_text'].str.lower().str.contains(kw[constraints['type']], na=False, regex=True)]
        valid &= set(m['parent_asin'].unique())
    
    # Wireless filter
    if constraints.get('wireless'):
        m = audio_df[audio_df['chunk_text'].str.lower().str.contains('wireless|bluetooth', na=False)]
        valid &= set(m['parent_asin'].unique())
    
    # RAG search
    q_vec = model.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, 200)
    
    # Group by product
    products = defaultdict(lambda: {'chunks': [], 'scores': [], 'aspects': []})
    for score, idx in zip(scores[0], indices[0]):
        row = audio_df.iloc[idx]
        pid = row['parent_asin']
        if pid in valid:
            products[pid]['chunks'].append(clean_text(row['chunk_text']))
            products[pid]['scores'].append(score)
            products[pid]['aspects'].append(row['aspect'])
            products[pid]['title'] = row.get('title', '')
            products[pid]['rating'] = row['rating']
    
    # Rank products
    ranked = []
    for pid, d in products.items():
        avg = np.mean(d['scores'])
        diversity = len(set(d['aspects'])) * 0.01
        ranked.append({
            'product_id': pid,
            'title': d['title'],
            'score': round(avg + diversity, 3),
            'num_chunks': len(d['chunks']),
            'aspects': list(set(d['aspects'])),
            'evidence': d['chunks'][:3]
        })
    
    ranked.sort(key=lambda x: x['score'], reverse=True)
    return ranked[:k], constraints

print("✓ Search engine ready!")

✓ Search engine ready!


In [5]:
queries = [
    "warm IEMs under 100 dollars with good bass for jazz",
    "comfortable headphones for office under 200",
    "best budget earbuds under 50 with clear sound",
    "noise canceling headphones for travel and commuting",
    "bassy IEMs for gym workout",
    "neutral headphones for studio monitoring",
]

for q in queries:
    print("="*65)
    print(f"🔍 {q}")
    results, constraints = search(q, k=3)
    print(f"📋 Constraints detected: {constraints}")
    print(f"📊 Found: {len(results)} products\n")
    
    for i, r in enumerate(results):
        print(f"  #{i+1} | Score: {r['score']} | {r['aspects']}")
        print(f"  Product: {str(r['title'])[:90]}")
        for j, ev in enumerate(r['evidence']):
            print(f"    └─ {ev[:130]}...")
        print()

🔍 warm IEMs under 100 dollars with good bass for jazz
📋 Constraints detected: {'max_price': 100, 'type': 'IEM', 'sound_signature': 'warm', 'use_case': 'jazz'}
📊 Found: 3 products

  #1 | Score: 0.6990000009536743 | ['bass']
  Product: Big upgrade for live playing - bass/vocals
    └─ Fantastic IEMs! I am a bass player and singer and these are exactly what I was looking for. I previously used a set of Audio Techn...

  #2 | Score: 0.6480000019073486 | ['treble']
  Product: Ridiculous sound quality for this price.
    └─ Ridiculous sound quality for this price. The iems come with memory foam buds but the replacements are silicon. Bass is ok, tight w...

  #3 | Score: 0.6460000276565552 | ['bass']
  Product: Great sound!
    └─ I looked at a lot of iems before I decided on these and I am very happy with my choice. OK, it takes a little bit to get used to p...

🔍 comfortable headphones for office under 200
📋 Constraints detected: {'max_price': 200, 'type': 'headphone', 'use_case': 'office'

In [6]:
import os

path = '/kaggle/input/datasets/vamshidharnarsingoji/audio-ids'
print(f"Is file: {os.path.isfile(path)}")
print(f"Is folder: {os.path.isdir(path)}")

if os.path.isdir(path):
    print(f"\nContents:")
    for f in os.listdir(path):
        print(f"  {f}")
elif os.path.isfile(path):
    import json
    with open(path, 'r') as f:
        data = json.load(f)
    print(f"\nType: {type(data).__name__}")
    print(f"Length: {len(data)}")
    print(f"Sample: {data[:5] if isinstance(data, list) else list(data.items())[:5]}")

Is file: False
Is folder: True

Contents:
  audio_ids.json


In [7]:
import json

path = '/kaggle/input/datasets/vamshidharnarsingoji/audio-ids/audio_ids.json'

with open(path, 'r') as f:
    data = json.load(f)

print(f"Type: {type(data).__name__}")
print(f"Length: {len(data)}")

if isinstance(data, list):
    print(f"\nIt's a LIST of IDs")
    print(f"Sample: {data[:5]}")
elif isinstance(data, dict):
    print(f"\nIt's a DICT")
    for i, (k, v) in enumerate(data.items()):
        print(f"  {k}: {v}")
        if i >= 4: break

Type: list
Length: 315586

It's a LIST of IDs
Sample: ['B000LPBW0G', 'B004I0G07U', 'B000GU77UK', 'B07X8R98R3', 'B00A7WA2OU']


In [8]:
from huggingface_hub import snapshot_download
import os
import glob

print("Downloading Electronics item metadata...")

download_dir = snapshot_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    allow_patterns=["*meta_Electronics*"]
)

# Find the file
meta_files = glob.glob(os.path.join(download_dir, "**", "*.jsonl"), recursive=True)

if not meta_files:
    # Check folder structure
    for root, dirs, files in os.walk(download_dir):
        for f in files:
            meta_files.append(os.path.join(root, f))

print(f"\nFound {len(meta_files)} metadata files:")
for f in meta_files:
    size = os.path.getsize(f) / 1024 / 1024
    print(f"  {size:.0f} MB - {os.path.basename(f)}")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


Found 1 metadata files:
  5003 MB - meta_Electronics.jsonl


In [9]:
import json

# Load your audio IDs
with open('/kaggle/input/datasets/vamshidharnarsingoji/audio-ids/audio_ids.json', 'r') as f:
    audio_ids = set(json.load(f))

print(f"Looking for {len(audio_ids):,} audio products...")

# Scan metadata
META_FILE = meta_files[0]  # The 5GB file
product_info = {}
found = 0

with open(META_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            item = json.loads(line.strip())
            pid = item.get('parent_asin', '')
            if pid in audio_ids:
                product_info[pid] = {
                    'title': item.get('title', ''),
                    'price': item.get('price', ''),
                    'average_rating': item.get('average_rating', ''),
                    'rating_number': item.get('rating_number', ''),
                    'store': item.get('store', ''),
                    'main_category': item.get('main_category', ''),
                }
                found += 1
                if found % 50000 == 0:
                    print(f"  Found {found:,}...")
        except:
            pass

print(f"\n✓ Matched {len(product_info):,} products")

Looking for 315,586 audio products...
  Found 50,000...
  Found 100,000...
  Found 150,000...
  Found 200,000...
  Found 250,000...
  Found 300,000...

✓ Matched 315,586 products


In [11]:
import pandas as pd

# Save product names to working directory
products_df = pd.DataFrame([
    {'parent_asin': pid, **info}
    for pid, info in product_info.items()
])
products_df.to_csv('/kaggle/working/audio_product_names.csv', index=False)

# Load chunks
INPUT_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/embeddings'
audio_df = pd.read_csv(f'{INPUT_DIR}/chunk_metadata.csv')

# Merge
audio_df = audio_df.merge(
    products_df[['parent_asin', 'title', 'price']],
    on='parent_asin',
    how='left',
    suffixes=('_old', '')
)

audio_df['title'] = audio_df['title'].fillna(audio_df['title_old'])

# Save to WORKING directory (writable)
audio_df.to_csv('/kaggle/working/chunk_metadata_updated.csv', index_label='chunk_id')

print(f"✓ Updated! Sample products:")
for _, row in audio_df[['title', 'price']].drop_duplicates().head(10).iterrows():
    print(f"  {str(row['title'])[:80]} | ${row['price']}")

✓ Updated! Sample products:
  National Geographic Kids Why?: Over 1,111 Answers to Everything | $14.91
  Unfreedom of the Press | $11.91
  Maxell CD-330 CD-to-Cassette Audio Adapter (190038) | $None
  Sony ICF-S79V Weather Band Shower Radio (Discontinued by Manufacturer) | $None
  Alphasonik 72887 Bluetooth Headphones, Best Wireless Noise Isolating Headset Swe | $19.95
  Uniden HS910 Headset for Cordless Phones | $49.95
  Panasonic PV-9450 4-Head Hi-Fi VCR | $114.95
  Yamaha RH1C Portable Headphones, Black | $14.95
  Motorola 53727 Earpiece with Microphone, Black | $12.69
  Midland 75-822 40 Channel CB-Way Radio | $124.95


In [12]:
# Change this line in your search setup:
audio_df = pd.read_csv('/kaggle/working/chunk_metadata_updated.csv')

In [13]:
import pandas as pd
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# Load updated metadata with real product names
INPUT_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/embeddings'
audio_df = pd.read_csv('/kaggle/working/chunk_metadata_updated.csv')

# Load model and index
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
index = faiss.read_index(f'{INPUT_DIR}/faiss_audio_index.bin')

print(f"✓ Loaded: {index.ntotal:,} vectors, {len(audio_df):,} chunks")
print(f"✓ Products with names: {audio_df['title'].notna().sum():,}")

def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    return ' '.join(text.split())

def parse_query(query):
    constraints = {}
    q = query.lower()
    m = re.search(r'(?:under|below|less than|budget|max|upto)\s*\$?(\d+)', q)
    if m: constraints['max_price'] = int(m.group(1))
    if any(w in q for w in ['iem', 'in-ear', 'in ear']): constraints['type'] = 'IEM'
    elif any(w in q for w in ['headphone', 'headset', 'over-ear']): constraints['type'] = 'headphone'
    elif any(w in q for w in ['earbud', 'tws', 'true wireless']): constraints['type'] = 'earbud'
    for sig in ['warm', 'neutral', 'bright', 'bassy', 'v-shaped', 'dark', 'analytical']:
        if sig in q: constraints['sound_signature'] = sig
    for use in ['gaming', 'studio', 'commuting', 'gym', 'workout', 'office', 'travel', 'jazz']:
        if use in q: constraints['use_case'] = use
    if any(w in q for w in ['noise cancel', 'anc', 'isolating']): constraints['noise_cancel'] = True
    if any(w in q for w in ['wireless', 'bluetooth']): constraints['wireless'] = True
    return constraints

def search(query, k=5):
    constraints = parse_query(query)
    valid = set(audio_df['parent_asin'].unique())
    
    # Type filter
    if 'type' in constraints:
        kw = {'IEM': 'iem|in-ear', 'headphone': 'headphone|headset|over-ear', 'earbud': 'earbud|tws|true wireless'}
        m = audio_df[audio_df['chunk_text'].str.lower().str.contains(kw[constraints['type']], na=False, regex=True)]
        valid &= set(m['parent_asin'].unique())
    
    # Price filter (NEW!)
    if 'max_price' in constraints:
        affordable = audio_df[
            (audio_df['price'].notna()) & 
            (audio_df['price'] != 'None') &
            (audio_df['price'].astype(float) <= constraints['max_price'])
        ]['parent_asin'].unique()
        if len(affordable) > 0:
            valid &= set(affordable)
    
    # Wireless filter
    if constraints.get('wireless'):
        m = audio_df[audio_df['chunk_text'].str.lower().str.contains('wireless|bluetooth', na=False)]
        valid &= set(m['parent_asin'].unique())
    
    # RAG search
    q_vec = model.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, 200)
    
    # Group by product
    products = defaultdict(lambda: {'chunks': [], 'scores': [], 'aspects': [], 'prices': []})
    for score, idx in zip(scores[0], indices[0]):
        row = audio_df.iloc[idx]
        pid = row['parent_asin']
        if pid in valid:
            products[pid]['chunks'].append(clean_text(row['chunk_text']))
            products[pid]['scores'].append(score)
            products[pid]['aspects'].append(row['aspect'])
            products[pid]['title'] = row.get('title', 'Unknown')
            products[pid]['price'] = row.get('price', 'N/A')
            products[pid]['rating'] = row['rating']
    
    # Rank
    ranked = []
    for pid, d in products.items():
        avg = np.mean(d['scores'])
        diversity = len(set(d['aspects'])) * 0.01
        ranked.append({
            'product_id': pid,
            'title': d['title'],
            'price': d['price'],
            'score': round(avg + diversity, 3),
            'num_chunks': len(d['chunks']),
            'aspects': list(set(d['aspects'])),
            'evidence': d['chunks'][:2]
        })
    
    ranked.sort(key=lambda x: x['score'], reverse=True)
    return ranked[:k], constraints

print("✓ Search engine ready with REAL product names + price filtering!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded: 111,742 vectors, 111,742 chunks
✓ Products with names: 111,742
✓ Search engine ready with REAL product names + price filtering!


In [19]:
queries = [
    "best under 30",
    "comfortable headphones for office under 200",
    "best budget earbuds under 50",
    "noise canceling headphones for travel under 150",
]

for q in queries:
    print("="*65)
    print(f"🔍 {q}")
    results, constraints = search(q, k=3)
    print(f"📋 Constraints: {constraints}")
    print(f"📊 Found: {len(results)} products\n")
    
    for i, r in enumerate(results):
        print(f"  #{i+1} | Score: {r['score']} | 💰 {r['price']}")
        print(f"  Product: {str(r['title'])[:90]}")
        print(f"  Aspects: {r['aspects']}")
        for j, ev in enumerate(r['evidence']):
            print(f"    └─ {ev[:130]}...")
        print()

🔍 best under 30
📋 Constraints: {'max_price': 30}
📊 Found: 3 products

  #1 | Score: 0.4099999964237213 | 💰 16.99
  Product: Apple EarPods Headphones with Lightning Connector. Microphone with Built-in Remote to Cont
  Aspects: ['comparison', 'comfort']
    └─ Better than paying 30 at target...
    └─ Great as usually, perfect soung, perfect fit...

  #2 | Score: 0.37400001287460327 | 💰 13.99
  Product: gorsun Kids Headphones, Lightweight Stereo Wired Children's Headsets for Kids Adults Adjus
  Aspects: ['mids', 'comfort']
    └─ I got these for my 8 year old son for his birthday. They are decent for the price and just fine for a kid his age. He is a little ...
    └─ Not bad for price but definitely kid size. My kid commented how small they fit over the ears. Good for spare...

  #3 | Score: 0.3659999966621399 | 💰 7.99
  Product: Two Pairs of 80mm Foam Earpads fits Infrared Wireless Headphones in GM Ford Toyota Nissan 
  Aspects: ['comfort']
    └─ Best price and surprisingly comfortabl

In [1]:
# ============================================
# STEP 1: Load 25 chunk files
# ============================================
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

CHUNKS_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/chunks/chunks'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Take first 25 chunk files
chunk_files = sorted([f for f in os.listdir(CHUNKS_DIR) if f.endswith('.jsonl')])[:25]
print(f"Using {len(chunk_files)} chunk files (was 10)\n")

all_chunks = []
for f in tqdm(chunk_files, desc="Loading chunks"):
    with open(f'{CHUNKS_DIR}/{f}') as fh:
        for line in fh:
            all_chunks.append(json.loads(line))

df = pd.DataFrame(all_chunks)
print(f"Total chunks: {len(df):,}")

# Keep audio aspects only
AUDIO_ASPECTS = ['bass', 'treble', 'mids', 'soundstage', 'comfort', 'comparison']
audio_df = df[df['aspect'].isin(AUDIO_ASPECTS)].copy()

print(f"Audio chunks: {len(audio_df):,}")
print(f"Products: {audio_df['parent_asin'].nunique():,}")
print(f"\nAspect distribution:")
for aspect, count in audio_df['aspect'].value_counts().items():
    print(f"  {aspect}: {count:,}")

audio_df.to_json(f'{OUTPUT_DIR}/audio_chunks_25.json', orient='records')
print(f"\n✓ Saved audio chunks")

Using 25 chunk files (was 10)



Loading chunks: 100%|██████████| 25/25 [00:12<00:00,  2.07it/s]


Total chunks: 1,672,119
Audio chunks: 294,988
Products: 15,498

Aspect distribution:
  comfort: 147,923
  bass: 68,473
  comparison: 33,449
  treble: 26,059
  mids: 13,290
  soundstage: 5,794

✓ Saved audio chunks


In [2]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

OUTPUT_DIR = '/kaggle/working/embeddings'
audio_df = pd.read_json(f'{OUTPUT_DIR}/audio_chunks_25.json')
texts = audio_df['chunk_text'].tolist()
print(f"Embedding {len(texts):,} chunks on CPU...")
print(f"Estimated time: {len(texts)/4000:.0f} minutes\n")

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

BATCH_SIZE = 128
embeddings_list = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list).astype('float32')
np.save(f'{OUTPUT_DIR}/embeddings_25.npy', embeddings)
print(f"\n✓ Shape: {embeddings.shape}")

Embedding 294,988 chunks on CPU...
Estimated time: 74 minutes



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [3]:
# Check if model is already downloaded from earlier
import os

model_path = os.path.expanduser('~/.cache/torch/sentence_transformers/sentence-transformers_all-MiniLM-L6-v2')

if os.path.exists(model_path):
    print("✓ Model already cached!")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    print("✓ Model loaded from cache")
else:
    print("Model not cached. Trying again...")
    # Try with smaller timeout
    import os
    os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
    
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    print("✓ Model loaded")

Model not cached. Trying again...


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Model loaded


In [4]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

OUTPUT_DIR = '/kaggle/working/embeddings'
audio_df = pd.read_json(f'{OUTPUT_DIR}/audio_chunks_25.json')
texts = audio_df['chunk_text'].tolist()
print(f"Embedding {len(texts):,} chunks on CPU...")
print(f"Estimated time: {len(texts)/4000:.0f} minutes\n")

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

BATCH_SIZE = 128
embeddings_list = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list).astype('float32')
np.save(f'{OUTPUT_DIR}/embeddings_25.npy', embeddings)
print(f"\n✓ Shape: {embeddings.shape}")

Embedding 294,988 chunks on CPU...
Estimated time: 74 minutes



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

100%|██████████| 2305/2305 [1:28:59<00:00,  2.32s/it]



✓ Shape: (294988, 384)


In [5]:
print("hello")

hello


In [6]:
import zipfile
import os

# Create zip of all files in embeddings folder
OUTPUT_DIR = '/kaggle/working/embeddings'
ZIP_PATH = '/kaggle/working/embeddings_backup.zip'

with zipfile.ZipFile(ZIP_PATH, 'w') as zipf:
    for f in os.listdir(OUTPUT_DIR):
        file_path = os.path.join(OUTPUT_DIR, f)
        zipf.write(file_path, f)

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"✓ Zip created: {size_mb:.0f} MB")

# Create download link
from IPython.display import FileLink
FileLink(ZIP_PATH)

✓ Zip created: 542 MB


/kaggle/working/embeddings_backup.zip

In [10]:
print("hello")

hello


In [ ]:
import faiss

OUTPUT_DIR = '/kaggle/working/embeddings'

embeddings = np.load(f'{OUTPUT_DIR}/embeddings_25.npy')
print(f"Loaded: {embeddings.shape[0]:,} vectors")

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, f'{OUTPUT_DIR}/faiss_25.bin')
audio_df.to_csv(f'{OUTPUT_DIR}/chunk_metadata_25.csv', index_label='chunk_id')

print(f"✓ Index: {index.ntotal:,} vectors saved")

In [2]:
import json
import pandas as pd

# Load audio IDs
with open('/kaggle/input/datasets/vamshidharnarsingoji/audio-ids/audio_ids.json', 'r') as f:
    audio_ids = set(json.load(f))

# Load metadata file (still on disk from earlier download)
import glob
meta_files = glob.glob('/root/.cache/huggingface/hub/**/meta_Electronics.jsonl', recursive=True)

if not meta_files:
    print("Metadata file not found - need to redownload")
else:
    META_FILE = meta_files[0]
    product_info = {}
    found = 0
    
    with open(META_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            pid = item.get('parent_asin', '')
            if pid in audio_ids:
                product_info[pid] = {
                    'title': item.get('title', ''),
                    'price': item.get('price', ''),
                    'average_rating': item.get('average_rating', ''),
                    'rating_number': item.get('rating_number', ''),
                }
                found += 1
                if found % 50000 == 0: print(f"  Found {found:,}...")
    
    products_df = pd.DataFrame([{'parent_asin': pid, **info} for pid, info in product_info.items()])
    products_df.to_csv('/kaggle/working/audio_product_names.csv', index=False)
    print(f"✓ Saved {len(products_df):,} product names")

Metadata file not found - need to redownload


In [4]:
from huggingface_hub import snapshot_download
import glob
import json
import pandas as pd

# Download metadata
print("Downloading Electronics metadata (5GB)...")
download_dir = snapshot_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    allow_patterns=["*meta_Electronics*"]
)

# Find file
meta_files = glob.glob(f'{download_dir}/**/meta_Electronics.jsonl', recursive=True)
print(f"Found: {meta_files[0]}")

# Load audio IDs
with open('/kaggle/input/datasets/vamshidharnarsingoji/audio-ids/audio_ids.json', 'r') as f:
    audio_ids = set(json.load(f))

# Extract everything
print(f"Extracting metadata for {len(audio_ids):,} products...")
product_info = {}
found = 0

with open(meta_files[0], 'r', encoding='utf-8') as f:
    for line in f:
        try:
            item = json.loads(line.strip())
            pid = item.get('parent_asin', '')
            if pid in audio_ids:
                product_info[pid] = {
                    'parent_asin': pid,
                    'title': item.get('title', ''),
                    'price': item.get('price', ''),
                    'average_rating': item.get('average_rating', ''),
                    'rating_number': item.get('rating_number', ''),
                    'store': item.get('store', ''),
                    'main_category': item.get('main_category', ''),
                    'categories': str(item.get('categories', [])),
                    'features': str(item.get('features', [])),
                    'description': str(item.get('description', [])),
                    'details': str(item.get('details', '')),
                    'bought_together': str(item.get('bought_together', '')),
                }
                found += 1
                if found % 50000 == 0:
                    print(f"  Found {found:,}...")
        except:
            pass

print(f"✓ Matched {len(product_info):,} products")

# Create mapping CSV
mapping_df = pd.DataFrame(list(product_info.values()))
mapping_df.to_csv('/kaggle/working/product_mapping.csv', index=False)

print(f"✓ Saved /kaggle/working/product_mapping.csv")
print(f"  Columns: {list(mapping_df.columns)}")
print(f"  Rows: {len(mapping_df):,}")
print(f"\nSample:")
print(mapping_df[['parent_asin', 'title', 'price', 'rating_number', 'store']].head(10))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [5]:
print("hello")

hello


In [6]:
# Test internet
import requests
try:
    r = requests.get('https://huggingface.co', timeout=10)
    print(f"Internet: OK (status {r.status_code})")
except:
    print("❌ No internet - check Kaggle settings")

# Check if metadata already exists from earlier
import os
for root, dirs, files in os.walk('/root/.cache'):
    for f in files:
        if 'meta_Electronics' in f:
            print(f"Found cached: {os.path.join(root, f)}")

Internet: OK (status 200)


In [7]:
from huggingface_hub import hf_hub_download
import os

# Download single file instead of whole folder
try:
    path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        filename="raw/meta_categories/meta_Electronics.jsonl",
        repo_type="dataset"
    )
    print(f"✓ Downloaded: {path}")
except:
    # Try alternative path
    path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023", 
        filename="meta_Electronics.jsonl",
        repo_type="dataset"
    )
    print(f"✓ Downloaded: {path}")

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6a2d21ef-17228a5f066f7b8d511a5d07;9b01a0eb-a647-48c4-b827-f4da5d32a1a9)

Entry Not Found for url: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/meta_Electronics.jsonl.

In [1]:
from huggingface_hub import hf_hub_download
import os

# Download single file instead of whole folder
try:
    path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        filename="raw/meta_categories/meta_Electronics.jsonl",
        repo_type="dataset"
    )
    print(f"✓ Downloaded: {path}")
except:
    # Try alternative path
    path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023", 
        filename="meta_Electronics.jsonl",
        repo_type="dataset"
    )
    print(f"✓ Downloaded: {path}")

raw/meta_categories/meta_Electronics.jso(…):   0%|          | 0.00/5.25G [00:00<?, ?B/s]

✓ Downloaded: /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw/meta_categories/meta_Electronics.jsonl


In [2]:
import json
import pandas as pd
import os
import glob

# Find the downloaded metadata file
meta_file = None
for root, dirs, files in os.walk('/root/.cache/huggingface'):
    for f in files:
        if 'meta_Electronics' in f and f.endswith('.jsonl'):
            meta_file = os.path.join(root, f)
            break

if not meta_file:
    # Also check /kaggle
    for root, dirs, files in os.walk('/kaggle'):
        for f in files:
            if 'meta_Electronics' in f and f.endswith('.jsonl'):
                meta_file = os.path.join(root, f)
                break

print(f"Metadata file: {meta_file}")

# Load audio IDs
with open('/kaggle/input/datasets/vamshidharnarsingoji/audio-ids/audio_ids.json', 'r') as f:
    audio_ids = set(json.load(f))

print(f"Extracting names for {len(audio_ids):,} products...")

product_info = {}
found = 0

with open(meta_file, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            item = json.loads(line.strip())
            pid = item.get('parent_asin', '')
            if pid in audio_ids:
                product_info[pid] = {
                    'parent_asin': pid,
                    'title': item.get('title', ''),
                    'price': item.get('price', ''),
                    'average_rating': item.get('average_rating', ''),
                    'rating_number': item.get('rating_number', ''),
                    'store': item.get('store', ''),
                }
                found += 1
                if found % 50000 == 0:
                    print(f"  Found {found:,}...")
        except:
            pass

print(f"✓ Matched {len(product_info):,} products")

# Save
mapping_df = pd.DataFrame(list(product_info.values()))
mapping_df.to_csv('/kaggle/working/product_mapping.csv', index=False)
print(f"✓ Saved /kaggle/working/product_mapping.csv")
print(f"  Rows: {len(mapping_df):,}")
print(f"  Columns: {list(mapping_df.columns)}")

Metadata file: /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw/meta_categories/meta_Electronics.jsonl
Extracting names for 315,586 products...
  Found 50,000...
  Found 100,000...
  Found 150,000...
  Found 200,000...
  Found 250,000...
  Found 300,000...
✓ Matched 315,586 products
✓ Saved /kaggle/working/product_mapping.csv
  Rows: 315,586
  Columns: ['parent_asin', 'title', 'price', 'average_rating', 'rating_number', 'store']


In [3]:
!pip install -q sentence-transformers faiss-cpu

import pandas as pd
import numpy as np
import json
import os
import re
import faiss
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# ============================================
# PATHS
# ============================================
INPUT_DIR = '/kaggle/input/datasets/vamshidharnarsingoji/embeddings-25'
PRODUCT_FILE = '/kaggle/working/product_mapping.csv'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# LOAD DATA
# ============================================
print("Loading embeddings and chunks...")
embeddings = np.load(f'{INPUT_DIR}/embedding_25.npy')
audio_df = pd.read_json(f'{INPUT_DIR}/audio_chunks_25.json')
print(f"  Chunks: {len(audio_df):,}")
print(f"  Products: {audio_df['parent_asin'].nunique():,}")

# ============================================
# BUILD FAISS INDEX
# ============================================
print("\nBuilding FAISS index...")
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, f'{OUTPUT_DIR}/faiss_25.bin')
print(f"✓ Index: {index.ntotal:,} vectors")

# ============================================
# MERGE PRODUCT NAMES
# ============================================
print("\nMerging product names...")
products_df = pd.read_csv(PRODUCT_FILE)
audio_df = audio_df.merge(
    products_df[['parent_asin', 'title', 'price', 'average_rating', 'rating_number']],
    on='parent_asin', how='left', suffixes=('_old', '')
)
audio_df['title'] = audio_df['title'].fillna(audio_df['title_old'])
audio_df.to_csv(f'{OUTPUT_DIR}/chunk_metadata_25.csv', index_label='chunk_id')
print(f"✓ Names merged: {audio_df['title'].notna().sum():,} chunks with real names")

# ============================================
# SEARCH ENGINE
# ============================================
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    return ' '.join(text.split())

def parse_query(query):
    c = {}
    q = query.lower()
    m = re.search(r'(?:under|below|less than|budget|max|upto)\s*\$?(\d+)', q)
    if m: c['max_price'] = int(m.group(1))
    if any(w in q for w in ['iem', 'in-ear']): c['type'] = 'IEM'
    elif any(w in q for w in ['headphone', 'headset', 'over-ear']): c['type'] = 'headphone'
    elif any(w in q for w in ['earbud', 'tws', 'true wireless']): c['type'] = 'earbud'
    for sig in ['warm', 'neutral', 'bright', 'bassy', 'v-shaped']:
        if sig in q: c['sound_signature'] = sig
    for use in ['gaming', 'studio', 'commuting', 'gym', 'workout', 'office', 'travel', 'jazz']:
        if use in q: c['use_case'] = use
    if any(w in q for w in ['noise cancel', 'anc']): c['noise_cancel'] = True
    if any(w in q for w in ['wireless', 'bluetooth']): c['wireless'] = True
    return c

def search(query, k=5):
    constraints = parse_query(query)
    valid = set(audio_df['parent_asin'].unique())
    
    # Exclude junk
    EXCLUDE = ['case', 'pad', 'replacement', 'cable', 'adapter', 'cover', 'pouch', 
               'charger', 'mount', 'stand', 'holder', 'kids', 'children']
    for w in EXCLUDE:
        bad = audio_df[audio_df['title'].str.lower().str.contains(w, na=False)]['parent_asin'].unique()
        valid -= set(bad)
    
    # Type filter
    if 'type' in constraints:
        kw = {'IEM': 'iem|in-ear', 'headphone': 'headphone|headset|over-ear', 'earbud': 'earbud|tws'}
        m = audio_df[audio_df['chunk_text'].str.lower().str.contains(kw[constraints['type']], na=False, regex=True)]
        valid &= set(m['parent_asin'].unique())
    
    # Price filter
    if 'max_price' in constraints:
        try:
            affordable = audio_df[(audio_df['price'].notna()) & (audio_df['price'] != 'None') & 
                                  (audio_df['price'].astype(float) <= constraints['max_price'])]['parent_asin'].unique()
            if len(affordable) > 0: valid &= set(affordable)
        except: pass
    
    # Search
    q_vec = model.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, 300)
    
    # Group
    products = defaultdict(lambda: {'chunks': [], 'scores': [], 'aspects': [], 'ratings': []})
    for score, idx in zip(scores[0], indices[0]):
        row = audio_df.iloc[idx]
        pid = row['parent_asin']
        if pid not in valid: continue
        if row.get('sentiment') == 'positive': score *= 1.1
        elif row.get('sentiment') == 'negative': score *= 0.5
        if row.get('rating', 5) < 3: continue
        
        products[pid]['chunks'].append(clean_text(row['chunk_text']))
        products[pid]['scores'].append(score)
        products[pid]['aspects'].append(row['aspect'])
        products[pid]['ratings'].append(row.get('rating', 0))
        products[pid]['title'] = row.get('title', 'Unknown')
        products[pid]['price'] = row.get('price', 'N/A')
    
    # Rank
    ranked = []
    for pid, d in products.items():
        if len(d['chunks']) < 2: continue
        avg_score = np.mean(d['scores'])
        avg_rating = np.mean(d['ratings'])
        diversity = len(set(d['aspects'])) * 0.01
        rating_boost = (avg_rating - 3.5) * 0.02
        
        ranked.append({
            'title': d['title'],
            'price': d['price'],
            'score': round(avg_score + diversity + rating_boost, 3),
            'aspects': list(set(d['aspects'])),
            'num_chunks': len(d['chunks']),
            'evidence': d['chunks'][:2]
        })
    
    ranked.sort(key=lambda x: x['score'], reverse=True)
    return ranked[:k], constraints

print("\n✓ Done! Ready for queries.")
print(f"   Products: {audio_df['parent_asin'].nunique():,}")
print(f"   Index: {index.ntotal:,} vectors")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 40.3 MB/s eta 0:00:0000:0100:01
Loading embeddings and chunks...
  Chunks: 294,988
  Products: 15,498

Building FAISS index...
✓ Index: 294,988 vectors

Merging product names...
✓ Names merged: 294,988 chunks with real names


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✓ Done! Ready for queries.
   Products: 15,498
   Index: 294,988 vectors


In [8]:
queries = [
    "best eaphones under 100",
    "iems with neutral tone above 20 and under 50",
    "comfortable headphones for office",
    "noise canceling headphones for travel under 150",
]

for q in queries:
    print("="*65)
    print(f"🔍 {q}")
    results, c = search(q, k=3)
    print(f"📋 {c}")
    for i, r in enumerate(results):
        print(f"  #{i+1} [{r['score']}] 💰${r['price']} | {str(r['title'])[:80]}")
        for ev in r['evidence']:
            print(f"    └─ {ev[:130]}...")
        print()

🔍 best eaphones under 100
📋 {'max_price': 100}
  #1 [0.695] 💰$48.2 | Sennheiser CX300 II CX 300 II Precision Enhanced Bass Earbuds, Black (Discontinu
    └─ Best ones in the market by far ,,, sound quality , volume and bass...
    └─ Theses are the best earbuds for this price range with lots of bass...

  #2 [0.675] 💰$60.9 | Samsung Galaxy Buds True Wireless Earbuds - White (Renewed)
    └─ The best headphones I've heard so far and fit perfectly...
    └─ Best headphones I’ve purchased even better than AirPods & I use them on my iPhone XR! !...

  #3 [0.674] 💰$29.99 | Koss KPH30iCL On-Ear Headphones, in-Line Microphone and Touch Remote Control, D-
    └─ These are probably the best headphones under $60....but you can get slightly better sound under $100. They sound clean, detailed, ...
    └─ These may be the best headphones you can buy under $50. Lightweight, sturdy, comfortable, and outstanding sound quality. If you re...

🔍 iems with neutral tone above 20 and under 50
📋 {'max_price'